In [1]:
import warnings
warnings.filterwarnings("ignore")

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc

import scvi
import torch
import matplotlib.pyplot as plt
import anndata as ad
import scipy.sparse as sp
import spVIPESmulti
from pathlib import Path

np.random.seed(0)
torch.manual_seed(0)
sc.settings.set_figure_params(dpi=80, frameon=False)

print(f"spVIPESmulti  : {spVIPESmulti.__version__}")
print(f"scvi-tools: {scvi.__version__}")
print(f"scanpy   : {sc.__version__}")
print(f"torch    : {torch.__version__} (CUDA available: {torch.cuda.is_available()})")
print(f"anndata  : {ad.__version__}")

spVIPESmulti  : 1.0.0
scvi-tools: 1.4.2
scanpy   : 1.12.1
torch    : 2.11.0+cu128 (CUDA available: True)
anndata  : 0.12.11


In [2]:
root = Path("/exports/para-lipg-hpc/mdmanurung/spVIPESmulti/docs/notebooks/")
path_obs = root/"data/bcells_obs.csv"
path_rna = root/"data/bcells_rna.csv"

In [3]:
adata = ad.read_csv(path_rna, first_column_names=True)
obs = pd.read_csv(path_obs, index_col=0)
adata.obs = obs
adata.layers["counts"] = adata.X.copy()

In [4]:
adata

AnnData object with n_obs × n_vars = 20761 × 38605
    obs: 'nCount_RNA', 'nFeature_RNA', 'nCount_ADT', 'nFeature_ADT', 'nCount_HTO', 'nFeature_HTO', 'batch', 'cellhashr_id', 'percent_mito', 'percent_ribo', 'percent_mito_ribo', 'log10GenesPerUMI', 'percent_top50', 'percent_hemo', 'scdblfinder_score', 'scdblfinder_class', 'scran_size_factors', 'CTgene', 'CTnt', 'CTaa', 'CTstrict', 'clonalProportion', 'clonalFrequency', 'cloneSize', 'barcode', 'IGH', 'cdr3_aa1', 'cdr3_nt1', 'IGLC', 'cdr3_nt2', 'has_bcr', 'NANP', 'NJunction', 'qc_keep', 'RNA.weight', 'ADT.weight', 'wsnn_res.1', 'seurat_clusters', 'MG_nn_res.0.1', 'MG_res.0.1', 'MG_nn_res.0.2', 'MG_res.0.2', 'MG_nn_res.0.3', 'MG_res.0.3', 'MG_nn_res.0.4', 'MG_res.0.4', 'MG_nn_res.0.5', 'MG_res.0.5', 'MG_nn_res.0.6', 'MG_res.0.6', 'MG_nn_res.0.7', 'MG_res.0.7', 'MG_nn_res.0.8', 'MG_res.0.8', 'MG_nn_res.0.9', 'MG_res.0.9', 'MG_nn_res.1', 'MG_res.1', 'MG_nn_res.1.1', 'MG_res.1.1', 'MG_nn_res.1.2', 'MG_res.1.2', 'MG_nn_res.1.3', 'MG_res.1.3', 

In [5]:
print("\n".join(adata.obs.columns))

nCount_RNA
nFeature_RNA
nCount_ADT
nFeature_ADT
nCount_HTO
nFeature_HTO
batch
cellhashr_id
percent_mito
percent_ribo
percent_mito_ribo
log10GenesPerUMI
percent_top50
percent_hemo
scdblfinder_score
scdblfinder_class
scran_size_factors
CTgene
CTnt
CTaa
CTstrict
clonalProportion
clonalFrequency
cloneSize
barcode
IGH
cdr3_aa1
cdr3_nt1
IGLC
cdr3_nt2
has_bcr
NANP
NJunction
qc_keep
RNA.weight
ADT.weight
wsnn_res.1
seurat_clusters
MG_nn_res.0.1
MG_res.0.1
MG_nn_res.0.2
MG_res.0.2
MG_nn_res.0.3
MG_res.0.3
MG_nn_res.0.4
MG_res.0.4
MG_nn_res.0.5
MG_res.0.5
MG_nn_res.0.6
MG_res.0.6
MG_nn_res.0.7
MG_res.0.7
MG_nn_res.0.8
MG_res.0.8
MG_nn_res.0.9
MG_res.0.9
MG_nn_res.1
MG_res.1
MG_nn_res.1.1
MG_res.1.1
MG_nn_res.1.2
MG_res.1.2
MG_nn_res.1.3
MG_res.1.3
MG_nn_res.1.4
MG_res.1.4
MG_nn_res.1.5
MG_res.1.5
MG_nn_res.1.6
MG_res.1.6
MG_nn_res.1.7
MG_res.1.7
MG_nn_res.1.8
MG_res.1.8
MG_nn_res.1.9
MG_res.1.9
MG_nn_res.2
MG_res.2
MG_nn_res.2.1
MG_res.2.1
MG_nn_res.2.2
MG_res.2.2
MG_nn_res.2.3
MG_res.2.3
MG_nn_

In [6]:
adata.obs["subject_id"].value_counts()

subject_id
GA0817    6314
GA1160    3896
GA0746    3793
GA0825    3402
GA0888    2128
GA1333    1214
Name: count, dtype: int64

In [7]:
adata.obs["batch"].value_counts()

batch
batch3    9132
batch2    6333
batch1    5296
Name: count, dtype: int64

In [8]:
adata.obs["day"].value_counts()

day
T5    3853
T7    3400
T3    3390
T4    3311
T6    2908
T2    2274
T1    1611
Name: count, dtype: int64

In [9]:
adata.obs["cluster_label"].value_counts()

cluster_label
Naïve              5671
Classical          3531
Atypical           3389
Activated          2612
Transitional       2278
Pre-Plasmablast    1725
Activated MZ       1028
Plasmablast         527
Name: count, dtype: int64

In [10]:
adata.obs["antigen_specific"].value_counts()

antigen_specific
Negative    10783
CRXV         6898
NANP         2207
Njunc         873
Name: count, dtype: int64

In [11]:
adata.obs["antigen_class"] = (
    adata.obs["antigen_specific"]
    .astype(str)
    .str.strip()
    .apply(lambda x: "Negative" if x.lower() == "negative" else "Positive")
)
adata.obs["antigen_class"].value_counts()

antigen_class
Negative    10783
Positive     9978
Name: count, dtype: int64

In [12]:
sc.pp.highly_variable_genes(adata, n_top_genes=5000, flavor="seurat_v3", batch_key="batch")
adata = adata[:, adata.var["highly_variable"]].copy()

In [13]:
# Split into per-time-point AnnData objects
antigen_class = sorted(adata.obs["antigen_class"].unique())
adatas_dict = {}
for ac in antigen_class:
    sub = adata[adata.obs["antigen_class"] == ac].copy()
    # Clean up — prepare_adatas doesn't need extra obsm/uns
    sub.uns = {}
    sub.obsm = {}
    sub.layers = {}
    adatas_dict[ac] = sub
    print(f"  {ac}: {sub.shape}")

# Concatenate with spVIPESmulti
adata_spv = spVIPESmulti.data.prepare_adatas(adatas_dict)

print(f"\nConcatenated AnnData: {adata_spv.shape}")
print(f"Groups             : {list(adata_spv.uns['groups_mapping'].values())}")
print(f"Group sizes        : {[len(g) for g in adata_spv.uns['groups_obs_indices']]}")

  Negative: (10783, 5000)
  Positive: (9978, 5000)

Concatenated AnnData: (20761, 10000)
Groups             : ['Negative', 'Positive']
Group sizes        : [10783, 9978]


In [14]:
spVIPESmulti.model.spVIPESmulti.setup_anndata(
    adata_spv,
    groups_key="groups",
    label_key="cluster_label",
    batch_key="batch"
)

INFO     spVIPESmulti: === spVIPESmulti AnnData Setup ===                                                          
INFO     spVIPESmulti: Setting up with groups_key: 'groups'                                                        
INFO     spVIPESmulti: Labels: Using 'cluster_label' from adata.obs                                                
INFO     spVIPESmulti: --- Product of Experts (PoE) Configuration ---                                              
INFO     spVIPESmulti: Will use: Label-based PoE                                                                   


In [15]:
# Model hyperparameters
N_SHARED   = 16
N_PRIVATE  = 8
N_HIDDEN   = 128
DROPOUT    = 0.1
MAX_EPOCHS = 150
BATCH_SIZE = 256
KL_WARMUP  = 20


model_spv = spVIPESmulti.model.spVIPESmulti(
    adata_spv,
    n_hidden=N_HIDDEN,
    n_dimensions_shared=N_SHARED,
    n_dimensions_private=N_PRIVATE,
    dropout_rate=DROPOUT,
    disentangle_preset="no_contrastive",
    # use_nf_prior=True,
    # nf_type="NSF",               # "NSF" or "MAF"
    # nf_transforms=3,
    # nf_target="both",          # "shared", "private", or "both"
)

INFO     spVIPESmulti: The model has been initialized                                                              


In [ ]:
# Get group indices from uns (canonical approach matching prepare_adatas output)
group_indices_list = [list(map(int, g)) for g in adata_spv.uns["groups_obs_indices"]]
print("Group sizes:", [len(g) for g in group_indices_list])

# Train
model_spv.train(
    group_indices_list,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    train_size=0.9,
    early_stopping=True,
    n_epochs_kl_warmup=KL_WARMUP,
)


Group sizes: [10783, 9978]


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA L40S') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 25/150:  16%|█▌        | 24/150 [01:54<10:14,  4.88s/it, v_num=1, train_loss=1.76e+3]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(model.history["elbo_train"]["elbo_train"], label="ELBO (train)")
if "elbo_validation" in model.history:
    ax.plot(model.history["elbo_validation"]["elbo_validation"], label="ELBO (val)")
ax.set_xlabel("Epoch")
ax.set_ylabel("ELBO")
ax.set_title("spVIPESmulti training curve")
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()